# Z-Image-Turbo on Kaggle T4×2 GPU — Production REST Demo

**English**

This notebook is the reproducible Kaggle entry point for running `Tongyi-MAI/Z-Image-Turbo` as an **API-only** image-generation service on exactly **two NVIDIA Tesla T4 16 GB GPUs**. One resident BF16 `ZImagePipeline` spans both GPUs: the transformer is placed on `cuda:0`, while the text encoder and VAE are placed on `cuda:1`. This is one logical pipeline, not two independent replicas and not a 2× speed claim.

The workflow validates the attached Kaggle model, reconstructs the audited server payload, starts an internal worker plus an authenticated FastAPI coordinator, and executes deterministic acceptance checks for the qualified `safe_512` and `high_768` profiles.

This is an independent engineering qualification project, not an official Tongyi-MAI release and not an SLA-backed hosted service.

**Tiếng Việt**

Notebook này là entry point có thể tái lập trên Kaggle để chạy `Tongyi-MAI/Z-Image-Turbo` dưới dạng dịch vụ sinh ảnh **API-only** trên đúng **hai GPU NVIDIA Tesla T4 16 GB**. Một `ZImagePipeline` BF16 resident trải trên cả hai GPU: transformer nằm trên `cuda:0`, còn text encoder và VAE nằm trên `cuda:1`. Đây là một logical pipeline, không phải hai replica độc lập và không phải tuyên bố tăng tốc 2×.

Workflow xác minh Kaggle model đã attach, khôi phục server payload đã audit, khởi động internal worker cùng FastAPI coordinator có xác thực, rồi chạy các acceptance check xác định cho hai profile đã qualification là `safe_512` và `high_768`.

Đây là dự án kiểm định kỹ thuật độc lập, không phải release chính thức của Tongyi-MAI và không phải dịch vụ hosted có SLA.


## Before you run / Trước khi chạy

**English**

Before running this notebook in Kaggle:

1. Select **Accelerator → GPU T4 x2**.
2. Attach Kaggle Model `dangkhoa2016/tongyi-mai-z-image-turbo`.
3. Select **PyTorch / default / Version 1**. The expected mount is `/kaggle/input/models/dangkhoa2016/tongyi-mai-z-image-turbo/pytorch/default/1`.
4. Keep **Internet = ON** only if you want to exercise the optional Cloudflare Quick Tunnel step. Local qualification uses loopback HTTP and does not require the tunnel.
5. Use a fresh Kaggle session and run the notebook from top to bottom.
6. Do **not** copy the full model into `/tmp` or `/kaggle/working`; the model stays read-only under `/kaggle/input`.

Qualified public profiles:

| Profile | Resolution | Steps | Guidance | Role |
| --- | ---: | ---: | ---: | --- |
| `safe_512` | 512×512 | 9 | 0.0 | default deterministic acceptance path |
| `high_768` | 768×768 | 9 | 0.0 | qualified higher-resolution single-flight path |

Unsupported by design: FP16 fallback, CPU fallback, 1024×1024 generation, multiple concurrent GPU pipelines, and full model staging to `/tmp`.

**Tiếng Việt**

Trước khi chạy notebook trên Kaggle:

1. Chọn **Accelerator → GPU T4 x2**.
2. Attach Kaggle Model `dangkhoa2016/tongyi-mai-z-image-turbo`.
3. Chọn **PyTorch / default / Version 1**. Mount mong đợi là `/kaggle/input/models/dangkhoa2016/tongyi-mai-z-image-turbo/pytorch/default/1`.
4. Chỉ cần giữ **Internet = ON** nếu muốn chạy bước Cloudflare Quick Tunnel tùy chọn. Qualification local dùng loopback HTTP và không phụ thuộc tunnel.
5. Dùng fresh Kaggle session và chạy notebook từ trên xuống dưới.
6. **Không** copy toàn bộ model sang `/tmp` hoặc `/kaggle/working`; model được giữ read-only dưới `/kaggle/input`.

Hai public profile đã qualification là `safe_512` (512×512) và `high_768` (768×768), đều chạy 9 bước với guidance 0.0. Chủ động không hỗ trợ FP16 fallback, CPU fallback, 1024×1024, nhiều GPU pipeline chạy đồng thời hoặc staging toàn bộ model sang `/tmp`.


## Workflow map / Sơ đồ quy trình

```text
REST client
   |
   v
FastAPI coordinator :8090 (CPU, Bearer auth)
   |
   | bounded single-flight queue
   v
Internal worker :8101 (loopback only)
   |
   +-- Transformer ---------------- cuda:0
   +-- Text encoder ---------------- cuda:1
   +-- VAE ------------------------- cuda:1
   |
   v
One resident BF16 ZImagePipeline across T4 x2
```

**English**

The notebook intentionally separates **runtime qualification** from the optional **public transport** check. A temporary tunnel failure does not invalidate a healthy local model/runtime.

**Tiếng Việt**

Notebook chủ động tách **runtime qualification** khỏi bước kiểm tra **public transport** tùy chọn. Lỗi tunnel tạm thời không làm mất giá trị của một local model/runtime đang hoạt động đúng.


## Step 1 — Runtime and model preflight / Kiểm tra runtime và model

**English**

Verify that the live Kaggle session exposes exactly two Tesla T4 GPUs and that the attached model resolves to the expected `ZImagePipeline` before any service is started.

**Tiếng Việt**

Xác minh session Kaggle hiện tại có đúng hai Tesla T4 và model đã attach đúng loại `ZImagePipeline` trước khi khởi động service.

### Expected evidence / Kết quả cần thấy

**English**

`PREFLIGHT=PASS` plus the printed Python, PyTorch, CUDA, Diffusers, Transformers, Accelerate, GPU and model provenance JSON.

**Tiếng Việt**

`PREFLIGHT=PASS`, kèm JSON provenance/runtime in Python, PyTorch, CUDA, Diffusers, Transformers, Accelerate, GPU và model.


In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys
import torch, diffusers, transformers, accelerate

MODEL_PATH = Path('/kaggle/input/models/dangkhoa2016/tongyi-mai-z-image-turbo/pytorch/default/1')
WORKSPACE = Path('/kaggle/working/Z-Image-Turbo-Kaggle-T4x2-REST-Server')
EXPECTED_MODEL_INDEX = MODEL_PATH / 'model_index.json'
EXPECTED_GPUS = ['Tesla T4', 'Tesla T4']

assert torch.cuda.is_available(), 'CUDA is required'
assert torch.cuda.device_count() == 2, f'exactly 2 GPUs required, got {torch.cuda.device_count()}'
gpus = [torch.cuda.get_device_name(i) for i in range(2)]
assert all('T4' in name for name in gpus), f'T4x2 required, got {gpus}'
assert MODEL_PATH.is_dir() and EXPECTED_MODEL_INDEX.is_file(), 'attached Kaggle model missing/wrong variation'
model_index = json.loads(EXPECTED_MODEL_INDEX.read_text())
assert model_index.get('_class_name') == 'ZImagePipeline', model_index.get('_class_name')

runtime = {
    'python': sys.version.split()[0], 'torch': torch.__version__,
    'cuda': torch.version.cuda, 'diffusers': diffusers.__version__,
    'transformers': transformers.__version__, 'accelerate': accelerate.__version__,
    'gpus': gpus, 'model_path': str(MODEL_PATH),
    'upstream_revision': 'f332072aa78be7aecdf3ee76d5c247082da564a6',
}
print(json.dumps(runtime, indent=2))
print('PREFLIGHT=PASS')


## Step 2 — Materialize the audited API-only source / Khôi phục source API-only đã audit

**English**

Reconstruct the exact embedded server/test payload in `/kaggle/working` and verify both the archive digest and every allow-listed file before writing it.

**Tiếng Việt**

Khôi phục chính xác server/test payload được nhúng vào `/kaggle/working`, đồng thời xác minh digest của archive và từng file trong allow-list trước khi ghi.

### Expected evidence / Kết quả cần thấy

**English**

`BOOTSTRAP_FILES=<count>` followed by `BOOTSTRAP_INTEGRITY=PASS`.

**Tiếng Việt**

`BOOTSTRAP_FILES=<count>`, sau đó là `BOOTSTRAP_INTEGRITY=PASS`.


In [ ]:
import base64, hashlib, io, json, shutil, tarfile
from pathlib import Path

WORKSPACE = Path('/kaggle/working/Z-Image-Turbo-Kaggle-T4x2-REST-Server')
PAYLOAD_SHA256 = '0ef66cfef4f27aa1303a93797589457e334e4b72e617368e51745a722fabac7a'
MANIFEST_SHA256 = '9604516f81a0b26f15635384d3b94954aac10ac72d98b81796776f11ab00d6fa'
MANIFEST = json.loads("[{\"bytes\":13,\"path\":\"src/zimage_server/__init__.py\",\"sha256\":\"75ad4f4c2943325f88064ad2beae890b59674be2bf0404b30afc5584dc4de4bb\"},{\"bytes\":4913,\"path\":\"src/zimage_server/app.py\",\"sha256\":\"cd91cc210e24c8fc1cab1cf3ef11dfdd8bb4eb9dc4dd82f5413aac92bb882c8b\"},{\"bytes\":347,\"path\":\"src/zimage_server/config.py\",\"sha256\":\"eed00972bfc315774cd2d71db20935ce1c6f0048d58fdcd2dc9137ba58b94a72\"},{\"bytes\":928,\"path\":\"src/zimage_server/internal_client.py\",\"sha256\":\"ab65fd699dc0c04d9f58a3686cc8732206a6a8a7cc3210fc85bd97e8a17d3606\"},{\"bytes\":3128,\"path\":\"src/zimage_server/jobs.py\",\"sha256\":\"ef3b335d6d129e9fbcaa77cd3716e329d009fdb67c0074121bf7ee92ec21c85b\"},{\"bytes\":4296,\"path\":\"src/zimage_server/runtime.py\",\"sha256\":\"675eafe52416c680e0a2f46474ba51c8eedc251aca5cc443c292c41110fe86d3\"},{\"bytes\":776,\"path\":\"src/zimage_server/validation.py\",\"sha256\":\"7355f1a3121f1e664f5fb0cf1422a0e20c8ad02f978d5959a5186a1e81692922\"},{\"bytes\":1639,\"path\":\"src/zimage_server/worker_api.py\",\"sha256\":\"ebc94057c0191d495baad119081b6f08a4a71d08d1af24f5978c93dea15ba9d2\"},{\"bytes\":3118,\"path\":\"scripts/endurance_queue_acceptance.py\",\"sha256\":\"1d5b550e212371523749339ec0cac76202e480ed2198713cf338ccc576ac9c65\"},{\"bytes\":1847,\"path\":\"scripts/error_boundary_acceptance.py\",\"sha256\":\"8ef6edeb6041f54d9496f804462b957a3c94cd2439ba06b0815891fe3bdf70f0\"},{\"bytes\":1884,\"path\":\"scripts/finalize_evidence.py\",\"sha256\":\"63591ba57bb24550a2027f2f40a57fa006f71f1019fca6d388a2dcedcc53886e\"},{\"bytes\":2027,\"path\":\"scripts/golden_rest_acceptance.py\",\"sha256\":\"89a35a33d171cdaccff691de517e6f6b621d7ff49ce4ecebafdaee554bdae6f3\"},{\"bytes\":1153,\"path\":\"scripts/local_control_acceptance.py\",\"sha256\":\"de226083874f29f42f1cca90ff7c7a9ed60f143ff6992b43a1622214a77350aa\"},{\"bytes\":2239,\"path\":\"scripts/mixed_profile_acceptance.py\",\"sha256\":\"801835dc008bece984a3a79eefef392ad67c2d0736d58f064182d69d04b88392\"},{\"bytes\":2563,\"path\":\"scripts/public_tunnel_acceptance.py\",\"sha256\":\"159fdb44bf2502b7f7dd2f31c9eb2601537d0418e9d8734c0d03bebee9d80787\"},{\"bytes\":1999,\"path\":\"scripts/qualify_mixed_existing.py\",\"sha256\":\"fe6e1802288b7e92161593a8ae11faf55009512d189e07285bccd7eacc5fb14c\"},{\"bytes\":1654,\"path\":\"scripts/start.sh\",\"sha256\":\"fd0cf4974dd5171e3e17363a16e42d2b3ca43765434df5c78dc0c64bba02dacf\"},{\"bytes\":403,\"path\":\"scripts/stop.sh\",\"sha256\":\"6075f5c3dd4c7591fb0e234a0d3b0ec178bd294d3b957bdd1fd91cea8fe2daef\"},{\"bytes\":264,\"path\":\"scripts/status.sh\",\"sha256\":\"adc8bdfddab5f53550151aeb5fd57980d4ced498c9382bcdd8e57d2c45d9a788\"},{\"bytes\":1022,\"path\":\"scripts/start_tunnel.sh\",\"sha256\":\"6e85fd13630818a2161d7a3265128e2cf264bfbc6d9a28bf5b820238ccb8d966\"},{\"bytes\":1118,\"path\":\"tests/test_api_jobs.py\",\"sha256\":\"d4c276f9d8fa223fd37722d942e7f595ec96c03cfda9cc58ecedb5a29836d7fe\"},{\"bytes\":1214,\"path\":\"tests/test_contract.py\",\"sha256\":\"a415472193557627d315154f76aa8a0cb0bec74063f8de776f574b39202dadf9\"},{\"bytes\":1132,\"path\":\"tests/test_jobs.py\",\"sha256\":\"c2b664ff7e0e1973901f3b2b436fdd293ac205a799a8d7d2735eae662ef60777\"},{\"bytes\":488,\"path\":\"tests/test_profiles.py\",\"sha256\":\"70d0ca794d43ba888aa760810fcf8c401a87398e12c080bef85c12d854a6fb53\"},{\"bytes\":759,\"path\":\"tests/test_root_metadata.py\",\"sha256\":\"8bbeb036fbe0eab31e083ff6e120788fc8ed16e74d34c21fb42dcc2b2f25603f\"},{\"bytes\":920,\"path\":\"tests/test_validation.py\",\"sha256\":\"7d290a312d25d13e3ae2cbca06f2373ab45adbada110ef51a9e91086059274fc\"},{\"bytes\":39,\"path\":\"pytest.ini\",\"sha256\":\"0922d82a60b616ffaf801784738325d7f1925ec773c99135f64767d14166d725\"},{\"bytes\":661,\"path\":\"pyproject.toml\",\"sha256\":\"9c69d2e06ee1c766048e629349901798781df76063df886cc41cb94c9133d846\"}]")
PAYLOAD_B64 = "H4sIAK5Wx2oC/+09a3fbRq79zF/B8u6uqFaSRcmPRFvlrJMoqbeO7Tpyd1tFh4cWKYs1Raok5djx+r9fADNDDh96OHHcbFb8YIscDOaNATAAJgpHWx/cqXXhmJETXjnhlmm6vhubZmN2883DPE14dre36T88+f/NprGT/sbvRnun2f5GbX7zCM88iq0Qiv/mf/MxTcvzTFPtqoOh8s3m+V97osL6t2azh1v6a6x/w9jeNXLrv7XT3qz/R3nc6SwIY3UytUYK/x1EyjgMpurMiieee67yzyfwyhJGgR8717GUZkU3/oh/nlo+zKZQYbBjC/p35grAl87M8e2opr6C7/snBzX1R8eynRD+9/snveuRM4vdwM/kbYRONAv8yIkElleu55zyj7yuN7blx+5IQDy3IudNYDselOQ6nl1Tx/jPvLI817biQFSvAZUeuxdJ9Xqv9s8O++bJ6fGrg8NeTeU/3nLo34PzpBL/DM7fsKbW1J/nztx5Nfe8XhgibmXkWVGkvnZ8J7SwQafOH3MnivWkXtWOosIzA7SzuKNGcQgkmKqqT13f9Bz/Ip50jZo6ta7F23bz6W6VskWOY3dU148h03ZLYBpDtwhUuZYoBPOPXCfoGitfY1j/QbWeOvEksOmD7YxVP/DNc8/yL00Gq488GD5AMWdlVdX6M/zP2oOPi5liBtKAFHemV9NUfELLjRz1FwSgDhP1UKewGinzuaNSobxmlMmJ56HP0C5pDnbCsvZc+sF73+SAazWGkqlWrp9MiFUNmvtUkJqtUbEdCtbJtObxRMc/Qeh+oAnDxvE/6lHgOzCcbJXoAGzNvbiLX6tUWfzF6uJcz5xR7NgAHUQNx79yw8BvXDixrv128Gb/dc+E9Wb2j3/qHWk1VeM14oOVKVsNwuLHBtLJOHrvQk21544VOqGqSePKuiCzinXIEs8jcwQTHiYvTGbbiS3X62pTN4pc/wKGmfDEwaXj8wpF89nMc6kZmfIHe52hXOOkubyySMFgNU9ngNG03QtcbQJXLYG+V4XbaYVdn2ZZvsJs+EahY8WOCRun/l1NpX4y3wfhpRN21PMg8KAp/XDu1NRw7gNBoNGDFxhS13eiiL/jQv8D6YgZuR+cZHXTKHNq2REdIJehWr7NMatuJE0IfBjZAlRO6FueOYLeALScgkEm8yIhUTWV4TOxYjdpNzHM3Ry0NJ15MxAk+Q2DksFG4HxnAMCUduq8T9i/fC90s6/VQtsdD8YQm8wpQtlGhAn0nda/545h57B8HYarIzpWmhY3SFPk5S9qDX2L8yzbv1KzGiMviBy9ymoC2JGgM/R67Mae09V+qx8gp1Xvz8PzQP3JurjwHLW/fd1ST3tv++pbYsBgccLfCPq4qzUbRqMJH0Slu+JHVbQXmDVa4lsA5fojb247IL+Z0WjiTK3uKwv6p5rQvjAIYhPooQXk0pKJMqdJt5l2acgQuiNH66jahzrxiPUYa16/pJrXY6i5VsvmiW9mlAH27LgOm3c+nTcNQXjjsul2MIowcYt+5BInjuXFE0rmP3MANNkonf3KJbv+OKDUK2OLfqfpd4UO5SWkvcc+lHWbxugGog4utSIqXhnAgtyP449cJ+oOOC+kE/WvDqVhQnC5nOwsLM7AVaRsRyZlfOXgXKal5NjS5uSOpUVcXpZos+hrRto4WuwC1/+daC10hMiElXFkAqFX8+wCgbD+YpirH99GwlXYcelrYWySqbDu6CC0PDh8h0f6d+tbU6DcV1YY6fxzVR3jBgXfawJSZiMabuxMoTfSruLDk+mrwrAIQsToXzoWJbTqrmS2rrWuVcFN4bwWP+WBZj/k2T4LItGliDXaSveLaFUH08aZjGar2Up7nGNx9JBx0Z0iY/0Ii4XPyoVLJRkT3I9FBj6ADzSvl1UnDm+yiEFWMV1ko8T+FM3Pp24sOrExRTHEtOdTYM9TNA5VIifQwPaJCfer+HbrqbQgQx0QAMtKDAn8LM5K14Y5xSoNsyylqLT929J8hI+iDfwLwoNEJCXg613ZWkcJbuuWFXO3/qKHDDrLxOSEP2G6AaGQBhMbxCqUmZEE9VH12G5up/UAzFSJcTD37aLsgqWs7FxGA+5BVxF808lJCQOxBobqt11VQ9HGc2JHEwIPwgi6EoFQeF/Cst18WqxLUkpaHVQDQZ+g+kdn9eLlDVnpwTyezWMTwZhcWdjaManhRiZJ3NVP6DFWlMqlx+K8lPVCOpYK8oRju5aJXCkIcTjDtmaQE9VBnoO7cpcqh784S81RwbwG6Y5x8ZJ8V1W+ev0v04g9pAp41fkPKntz5z8tY3ej/32Mh/ZjFAdJV5aqWJNPsAr+kbzoAP7B8bvI81e5kvOEKx5p9bx37XhCugt6nTjuxSRO36PYmUXp68XctS1/BFNvZKHqcuwFVgwF5pSXsAS1yBo75o7R0hTBOiO/rTA5VaR1RGV0qkcXvtV4HdhvKr8LdC9bcrfZaFaZLKhNANrc231SQAbfEmT0ezmyO+W/dP3nlEUPQAhWrH9jd7uZW/8wWJvzn8db/6Y5nsOu55hmcqDjw8bNBDdFnAv9HgW++D0PPc89bzh03JH9xllwRfnX8elPvVPz+f5bWsGTOJ51traM1l4DVkjD6DwxmobGVaiyslAnlaftjmJGUzKSDeqfcyU14DUAplKXC/w+1QGpsTt1gHXo7lRRkBGnSaVqDWxjA6iQrQuwqiKJRQmPUpCICmoRUr0BP0RdBO9j7Ra5EBKDGqaJHIdp3nXUW/hwhzoj6oeskhWZzw71RK5LeMuhW3NdIc6Zknrl+0TI0pLqC6l7lxqOwmCEhVYbwK4D+6Vr83hcf6JVU+gJnURE3VvtBapZ/bje5yo/C5XtI6r4FqLT7tJc7BCmq50cv+3zoqv3G1z+no5mu9n85PGUp3GDmNBScfc8sG+gr+FbA4cX5HnbkbunphKGqAujP/OsUea4hxjc07mP9WaHQ2MhgmCJNAEaiA0nA5Z0p0li8uPTfxTiHtYAYAX9323u5el/a89obuj/F0b/SREjXuIJU29dJB9geic7ATBEwlRgdpOwlI4zw3eWAsQQTwB52gvL86xzz0mOzrMKKF1eP+LonDGnDDw9UOokCgVhyKZHjjcWh2+dpKjBAOnpsKbyf4vO31IygHgapnwg5vPjpTSVMHRYXzWoEYN4DiL1IIpDXhQ/1R0CCglM0ogzTJZNci7sNd20txvPUVng2G+dqTWbBKGjZ6sNNN7II8IFzXYRuRLAPd/lAL1gdJkp7BA+FOrFkjNwffqlw0K6cOKuQBbMaqT47mqMwNShInWuPYZaWM40EJJEWQnswFmcp+GAcgUmG86ZdYN0XdoeM+f2ifYTpP4LU4N+wUnZwD/bQL8nznUC6gfvsTUwvRr4R5fVCqMgRCTraCdZ7WLHNq0YUgArqixZLeFdzH6df6rWVKFI6ajsBDjhFbJ6e9oV0xHK7nPpEA9Y7YZ0dID1LpywZKdVwxr9MXdhBp0jVhhGflqYxb+08GwFGrNgposuIguFEk1Pbl1rgpOhfkSl2hjStGrZqmqgmgm61YI5kJRT6NZqQSvEQBXpOEHMIVnZKBgsvjo763U/VxhKnVCmM5Qqk9QXc1YTbWLuHJtYYsstqWfC/3BpOccYssyW7blkNkKTGpZZEAe+OwLG+nuRPW3eBI+kCoA/JGgWN3hRU5eqSNOp8JNzwybBEhySEtRF7jrRTyar5a4EvaSllr9TIyMPRkAHCcTI82h91jPZOiXDwU73sd05u5vSw5cF0za7KjgTyqBwTWRx0AZXShl/D1xfF3xwS6okUd1FlWQjjQR3yRxmlcGRveeYytljK7o0bQDLIUkHp+SkKiHpUJfC0KVJONeTxV4GMxA0BXBqSA35+s/MhPx4rUXqyumtNEVht0GmAHXT98kc8k1juGAbSrsOt4tkqBj/Ud4XD9aeZMGtnXsMXFc0Wa9Fi3CIEwbay/BnJudKYfxBe4CRmUdtPiuSeDRhUFPUHQAFnDpRhGdstDNQ6l2mDKgLMLs3ixZrygqEjudYUVkVFy3rjeX813r+EzJR6wFVACvs/5ttIy//t7dbm/OfL03+n1gR+gOUqYNJ+F/oNJBABeGIuxDY7ng8hwmXnDf9RpaRJ+7MQZ6z3Do/Z4zP7b5xA+AA/Itj0oRWlDfHL3uH5sl+/0dxgK1tMRunLdcHtmyLbGCiLdvyLy4ngdVqGrtbwAFf3Lj1qeXWM9ZRW7MbasAWN8LeMkBMOT7rn5z1zdPj436hCBR1gRXYylh91pnVZx2tPuto9VlnVp9b7Gw7Apxvev39l/v9/YfAKow9AW3v3ye9F/3eS/Nl75eDFz3zzf4J22BCy4/GQTglo64mbCtoPGsyBTB+MuDTlYW7jHGXKFzYeHG1zAKdS151MoPRhSJJzMkkIP8C5AdG245KAWwHDdbMqTUTyUmJs9AZe3gol7K9WXGIS780do3R3LbQFMG6slzSAoGoE4RyIi9qFEDTSA5qlRksZPS52uuTM7O//e+Wedr7+ezgtPdSEmBxs8ZGDaQygL02eTmYrLvMStBFGQdG48LRW9Wh3ADLv9G1/rYmvBAwV2JYKD5E1ZU1HRer2mFWi9GdVrDbSJcPdpnthqyzMEmXltYWMCJkS+b6tnPdINV/dbm9R7b7GK6DI1xJbw7evj04el207bjVLmZz6i7Uu9D/miiYbE8YE5TWCxUstjBHPidh2dgVxrlkCY7HAgsmTTKrBLedTrOqbNUaMocLtmE64ZjNG+DKq6VTP0vlGkjHTMAMSxBebT0rQuZak0mkhnXZlBJty0EkS6arnVsenk/bOXtoVB1OnWkQ3nRvm9BJxvZr9zlI1ob0WxvN5tiBrSf4fpdFAEy15dEwR2bgezekzUtBqktXeUmfqXXRp0uWPwmASac2JmMzTc5M4XzWb7tqCQ1cY82cHZXlU29zBdyVTNrvvksmTjJb5X7AWZvvG5y3CVIBIBWjFA1jEzVR6Wmh3B80EZfYrpWtzKPjvnl4vP8yQ9eEMXM32ZZRfhkkRprDodQZQh5n0nhWQk99VfIkMFs/iX6CTAgUdOZYl3z+miiwRUBH0yM7B9A5iazMwKRVybsuQEU+w/xafNFZX3c1LKvTBFo2tfw5THQ0KtVdn5ReA2ZxOqzelxwQR5FQFRiN7KpnjnBd0ZXonDfMLjpugsL7ucFeazmlLRqsCAh6ywL48ylQ67ETOmS2QoYsApzesuA5AxcBmf2cyyJ6s5v8kghDg9mHD5rSLpdW574kAsceoLPcn05/E6N73gtqttvSYZEYucb0Evc6dCTz44jRNNW5dmG+Bpe5A4sMr3afjJLVJFRdZiO30FQgXSB3jZmsTqJWNSLrytElFDVcPFMr7monmb2T+cFBAZx3b0QTq7WzK2elE23z/CZ2Ir1KByPceS7FYo1jOvBasJQStdRt3hknWeQdiQbkfHJSo+4s+ajlfZHIvju/+nJQNMipu0LZ1NfYyEtAZStIo1UgAZWsCi07/yXgZQujhCcpeillVwK1O/ctX10aVzxmosHLpcomuoxNkr7k+3DmXsMmRYuKgOF/DoLTXEZiAYb9KAeiuQMw9D/fE0t2OdkzS/zSs5JRYaFwzvN96AIRQCFGl6xa2CxFJzkbVme3BewhiTh48CWsXPLbN9c9Mkt37Al3lPMmTlZDccct9akbIxd7695Bg28LujbN8pCjipOZLm15ojsFBOx1uYHja460OIsRCIAF+WlPXVaNhGVcWRVCtaQ+KabFdcqpM/MMgjRHNpq8h9L/pWqVh1IBrrL/3C7o/7abexv932PG/wCGcHaD5yj+jFsilnNTwo/eTA29pY+SuXeOILpjwbmgkQpIYnoWUwFJwVFfCq4w1uZ+4v5vA4/o4yFGBCQ1LUMIY1aInIs/a1gR/LRuWENQsQhzPda109fPhccK13oArBvhKQ7IVZCl2gBCpy+rjuYHfp1lUGnzjEQogxiFAdrlERPwELZwtZv7LuxbwNZ6QYi8KxJHKJh9JmAgihMLpIO6UVPbsFlZwEd2m9UGfQXGuZrGArDVH7qq0WiiciaLGL/DilpWedsRUiQbIV73wu6V8Fa5gVOKjFVuKCWQqevznQCbCG+6TPE12BPkZOs6l+xYmD3tUPyQBYHOIKZF4jK1TJdAauadwf3v7h9F+s9NsTHwzuPQ/7axvZen/4axt6H/X9j5TxAtNP6UDT4/OjZUGhKqGApqYXwncVLETy2zx0lcmaUopkjuZlNAijVFbAI5igOpGFxyFMwY5cErN+djJnoLTSbFMcm/aDGtEf0pG9apENFJyerRctGB2ImIwi2AQJwPRfNKPChYg7M2DpgJam+SglIrgFKbEVAgZib11aU4mSPEet4TZflzdhcC5FNsIxSlNEDMesFhcj2rpMFihJ/o0mgvB9ytik+JBUFdlLIAJMXgI8XAI/kevMviEo7OhfAiwEAUOv/bZPTWj5Yke1ibUvyNReFC+DLKzrCk0jyQROKswioux4HoZJfWI7WGs4i07Ffbz97LA5u7hJzPI+GQnlm1qQEcND4Tu0FZZf9W/h14RX2hbXRBE5Is/GQMMmZnBUMj1kOpWdFG0P5S+b9R6M7iaMvx7XlIOkvmTGCNcMbih0/nAlf6/+zs5v0/25v4n48q/xeMecp9Oheb+Cz28mw+bWoKhR38RFsWweNtAc9o8gh87PSCdL1VEWtS2T8jg59bbV8OIIh8nYhbCCSPanSn/PPt8ZH5Y2//Ze+UPMq/+w5zwwa12rsRPdHfnKCFjravjlzfmaKCWCVW2o3wJ/RSaIU3TGJFbyfrPLhyVEsdWd5UDUYgvqpWrEZzHzLArvjeCqfqVeDNpw40BfPTuTEwhCEWDTnD0cSNQbgGZp0iCHqsIDxjhq/IrtdU9GD3bvjG4tjCxTbxnkmiIKZ7yq04a+yorFFpRB38l4m9k7rb3wmudLEfauJ3ujAmVM75VGwrwgG1Wks8TuWhqmU9SqvKGu6jqeuoschzFEMgYGBEntBgu7Xa7aqtZktZ7Vs60OhEW+HsSmymcYMSzm3N3qJYOjhRhXW86Aacnw/V3iVesopLR7ADedqQBh5/pUp4o4lxPeHvbnWoSA4WRlVBjZhrYfTLQdIVLkcCPxAHlDFUeKdH86n+uywCSNb0LA/lYEirpNqqiVeBA1V1v+d9RlIXLYGuJtm13xVxKwo7+IlYVPRsfYWt2zIfm+1WU1nkcYGRdrqq3CGyjQY727yfy8s50IFLGUmhRs8WOPOU+b3IFcqOp7RAstXsyn4CZIaipOfCUQMoJ3AX5FauUGfOQufKDeZAqkbzMKSApL76AUg3z1ETWQdGZ8jXDS+YZ8i5TTzrJjhzBvg1VS+UBrWAuTZFwpyOL4yKKw78+BiHqbUK80bg8Wkpq2iVpCAVLoKMCMhKSR6bTJxay0lZp0HWr/K3DHDa5k4yBGEJoNwFHDLbK8rSw2aoaPFzpiLitBmbxH7LyRezeTM9HkzPiQE6e3Q8HBAs/k/PHJdgEofJKSL2ZR08xj1qZKyNaXmNjBUtK5y2LmlYDrZQpzVxGYtx3cHCkOjo9aC8+2m981UwaA4XQuGyusZFxWGrReTGWsiNtZAzzhKWOkaac1J5pr5DbplLbQNuNU76mayPa2AgbyP5HQLICkZMI/sIVv7dCsOCWUiWK5+rSFGA9vNZ76xnvu2dHuwfHvy23z84Puqe7L99m1RBe3XwS8/85/Fzs3f08ux0/+hFLwfwpvfm+PRX821///nB4UH/V2T1BMhGXv5q5X/c/c1zjC5gIcl4QPF/5fnP9m7+/Gen1Wpt5P8/Sf5fI7bT16YHWE/eZ2Id6aa5mJpIZfeV7j5VFub/S8Xg+4RUWkNGzNrDyXJx7YHCK3HsIg6ShDbSizGXYLNTRhNndBmxGCbs90DcKUFxZZneGwdK0mxosM3GWqrZMBapNWrqOsGtqknJ/HKIByp50WTlpdQjh3x262zmrzVz06rKN7mUVxUZyrVqyvQQAvHcx8s2ApJGjGZr+2P6gUIvUuYEveAZywcZOETkF7ebRh4uOyQJXDsPl+0PAddq5eGKzZNglfWWvQhHl0ZQW62frCqftoCpTJRjSxZqouwCkAE/IhuibwgqLJQCQ41Lty4YhFX8NOs3aBL7UeNHcKm8hG8fwTRLx2eI/fayo17hSCC7fFlTr5BjZkUmkfXLy04aXMpG905Pj0/N58dnRy/3gRfef/Gid9KXeeWvjv+jEzz3g2OKAX80/+9me6e9XfD/3tnYfz46/7eYq5M9kSsfxbVVqgryhdxwvyJmWUVREkldJlORrvcIrkSMr2S4vaoyda/J66qQmRLqsF3V93af4P/S7LP5OWyXZflZSj3GOCteXRKJytCwCJClbShSznzmi8CD7shlFn0lfLgr+MIgqVG4iW6Xt0lScDIFZQW2OLyVpdJRK+uNF9MJVSZWaL+HTQky3lbQA5c8seCtVVPpnUwREG3fiTxL7W9XuJNoJUlZz73+5Kafda8XVSCHGcQjHGaShMRbhGonObJXyJG9IjuyV4jdqFxZiMoQdRT3mlRkf4yKYHMILZm5wi8KIF1hJq3JK7kFwdtT7I2M1w9WodGUHGYrIrJ0BitFkk6xstc1sYpGsDkhKiw1g1Sz8I0BDMQHSXdYKSh6JfBimpxT8hOS8shfZWiYUmgNjsOIW2gl2wJmc5As9mwzsooySEvgBvk0uUQPvtCdExVSrZW1JtWySSiRzlSG69edqIy4UBBg6B36mq1AUaVKVsGSGyiJq6ks4GoYfRlUGHdTGab8TYpH4nOwm3gO+euQv95jbBgVNBkVBBj2zhMlvhi7uTKe4fJUK8gjX+MffJlZoeV5MC2QXqQiLyVxV3hzRtGfib1Nuox4ErNYuzuFEdY/5sCyjLmoUxcHM5welnOmHKic8awQ41kpYTyL2QRQ5dXB0f6h+fPZ/uHBq4MXqca18t/BJAr+j9MQvLbtYZV/a/B/ezv5+x+2t/eMDf+3sf/5OL2fZM7zUKq/JADE2x/3sZk7u05zPLKazT27bVtPt3dsZ29v78mT8x3r/InVtIy2cb7zZK/dtJ+2z3db2yNrd8fZG+3Cf8dqN/e0BcZDs0kQBxehNZuowVi11GgKlPNLNClay3Zou7XQcujRNKWLDIY+Ta+SnNEv160kYDlrCW6SoiRRiiVAZkGUCyYhWXMoeauSTzcpWk8T9TDqZGb/skhx/BEmMKn5S5npSxIuQn2mGk8KnnClMV+Lpi/KunYvMpwwG0ltJDCHTElqiwAV5pH1sSNLn8X1aw+nbkwqtWKYGRyFlGABPNmhAVPjiy6SgOSrpc+1d9dPnp4cvX4XvvPfXRvWO3/B8Xl5QO7MCNUkU5UFPV0rN4HJQZeZwwBl8y9YAyCDl/QPfanWimEScjhzhie1QsiEcnhuAVGqtnx9fPiyd2TiprmuzlLwfyyUEzrGhIH3wBzgCv6v1Woa+fPfZnPD/321578mv9TroZnBhMtMS5H4w8L5QsIvJua6/M5APDRy0NsQTY4yXj2clq5pXE6R+2Rc/Lre9e2mWX1yh76f7ZSXX3Mj6SBztDt37U21utYhMaL97KfCaZXKT4e5M92Q2d7K3nUCgqlJsCeho9Kz3Kz/XAa4kF5T+cV4ySnkOJCA0qumGVj+kFHUkR8ttprNPERZHRcfgvJKpujoauJ8qjEsHv4VTlOxHVK1CvuxOOYr7kkvjo/6p8eH5snh/lHvaz5J++8+/8toMR93/99u7RXsv5rG9mb/3+h//hv9vy7I8R8EMqg1hj28SVQ0f8xdB82fZ+g44lmXTkZTs0D1sr7uZoHfF/SantyT/sneX/zX1+f6tUqjQ7qLUl3Nan+gdnOxP9BHyPjl3mAPxwcuV9F8jZ5KpZ4+xRBITOcgh7Asroyi3w+toc/rcZNXNRT9WorAC11XREgkBSgHucAhBUl1tzXkAI0qXainp9cl0+eW+JyFbleZlxXebsqjhaKzFaBno4xpF7M5ftSZw0uNebXId9oLc7UER8G/BXJk/W1wvEvgE1+fQoaEJce6DVrDtfyMeCkR+ausAb9OGcY9y8jDF8z3ym1hlljwYWk4O+DfR9jpLcidSAU/Hrz+EadNiUAg3FAO/t17KW4AN/NOKRvp4YH5/8wp/+Py/ztAIIr8/86G//9y+P+P0wV++v0lsMK5YCGs4TQhBWj4wqcr1GiBKCDkjvLsX4ogkVdACuaOrgvsfJq2EAP93N59is6w9TGeITnNofSB14suQRmxvjEpgteDuI6kmkH2e0lpiqygrKlyaqKygwkYBkGc1YoifHrEK4WI4Aq2DKJuyYgnHACgHrAoZjzeAQxCHWalRohYMsqR7khAZGwm62xN1WNYNlquOWZR07mwxvm25Tjm4HIxcqEYLeBGfeTyTMLH4x7VKjgplAqxGXF8FGCUQS8rkD+c2C15s7QpFMafbe+wVFT+tDX/2Y0elkvRxpNmwfZh0Qwrs3ZIReWyGaeUSL5iqX8Wy4QyUfdeRgnrSriij3gI51I6t8KGYMVCLVA8yreFF1dQOg/LvOqUXzJa1zg/ijt7R8X1wfzfzeyx/0cf3a9hFlB6uo/n+twgFRc4SSF3SsJdCHGLuIul7gPaR1moLha5Fluoaidnzw8PXpj9s6Oj3uHmBGbzlMl/zJb6xmTnQHRpDYazfRz7X6MQ/3lnr7mR/x5R/rvwgtKLPYPoc8t3IVDl0JZCbdGVSLC5YpUa+EfHAL6CwiYXWuLLd5yUchkJv2dYE+JrmFRUIKHJro65WJRc4uOqtLG3iJdDjSZTYN7JIgdVWASAwuxJKxoR9I5+6dx0PWt6blvqdUe9Ti4rEhvoIIkERZrPkEWeIgS0gQ/ypQ/lnMkNSQsyy7pXWT2bfgdOa+yGUcyOjGoseitZ6hEaBW2G5zPTc4E7g8/Grvqdir4c/J+yUl1L2E3b8Wg86G0NZW09B7lETSsxcU2MBCcXCK9yA2RYbO56WuMM5HoVoV5cD3sW9MHRr9fMdZEnV/iswp0CFv0PYb4D+4+cUWetGcrOILTEhwuzlR+SFFcBz5uweAPpzGQhdBqPSzSDKtUpb3A2Klfa8BSZcT9kxlJkTZO5QsnhvcqHbHEcNFap++AxFuIR9ZFud1pRnfIuugcWoxRLnhUn37BSZvze5x98Bqfxtlj8SDrVYFPn7gF59D/tPETwfyScNaLJ5+AxkKnb29lZfP/Hbj7+s2G0N/7/j/L837db8yjcOnf9Lce/Us+taKJETqzWe848UNE9c2y5HrF63Y/k8ZTTs6P+wZteV/sLoknseDSF7stU6zMVUhiMpnIgL7iIkhdxN7p4T5lA/iFZ6srI5t80RQHmbjBQv1XrUVqAZDykDod/V9E6F2jJ7CaeBH4beJAffqic/FpZzPUmjDEa04xCJ46UWfcTAiOUmDWhB6pMl3g5DUpElQjuk3q73Hc1iBqjyTSw9VlNbQa7zWZVOfkV+D3qjm/Tho5QS1NynvP3RfrBStG0y2gaTJdbqSWKw6qmPtuynastf+55auvZ34ykk09+7f94fIS3WXejcKT6wWQ+Sys0VedXLhBWDAUr3UfUSO8j6uA1G/X6JIhiNakFfKBWYGXgN8ybuudcOR5emRmoz3Ae8SuNQCa4oAqpf4PKOKNJoP7lW/WZNDc43My1NeqxwqyQTshq+XMwEAa6pKBj94N8j0GAmbpS+voDQAGPzA9ilqlj1+1uUtMS2w3jG/Lzl24XxIaOUEDmr1/pwMSOIkVWJ7aqCtqmd7gG8u1NFDvT3rUb65V/HZ/+1Ds1T3v7L381sZuOz/qJj3QmVfg9w3z7DJOt+bTJjzLWm210Lgc0Rwf2pXT9V/Fymd8O3uy/7pn7JwemyED/tY+crXhxzNJpCs1YNE1HAbAVwAHFwRpzVQYWE5bGUjWSebuGc4/UfWxIx2qF98nb3ukvmaF95x8ev9g/NM9OD7vl4/POT10vun+N3vmV0p7fqBy/OP1fFAezz8T+reL/mrutvP1Hc2934//19fN/KIzjhV0oj7NjElUiayrbjv+u2gFyaK6Naq9uSk/+glkZ7aPND3g94vQ4ZIa/o/zpZiBAqprQAl66sH/UmyxNA9KbkEWGgwP0e6dvBMx//gP7+Nz5O0itzERjqtbHEnIFFWGKjXezlVPWt/3jEyCvP59BZ/VechJbUf6k9Y98w+ehAKvWf/H+X6NtbNb/V7P+BTvC1/8aXImmSOs5QzS2JJOvZIGrYn3xo05kUAT3Idi/RUiqGi3gzf5vhTE3/3xoKrBi/Rt7e3n9T2unvTn/+/r3/+cHR9J+PvKCuT32rBCNhA6PX3clVRBfsSgWrVYbSYL75xI82xnBM9U0XUNFoFUZ3mMEuIEzOASxL3RA8lehXoGAw+IiKO/CjSfz88YomEr9IHfJFr9MEegmhrqLt+zgvY+nnDJQ3XP9+XXdmtq721gyKoJUWHq8NKEGSigr73neucTVDIfq3/4msUM58VkCrZawSVBohzQJ+IPkVRhI5ISY+MwbzXm9et0P6tY8DuYzvGke3rGnSgdAfcYwLZeJpcpRU5G/NJG5/IseOX+ohtpuVjk7CSUhO3gRgsBc7wVqRQzEoG7VPzTrT4ffv2vAYKWd+w4Hp8JbpALvB+tCrRuCDaxKXKgPUJkNit/nyzapv0YVkV6sPO1JGXButyOL4SXbXJLNuXZjtcl4T1UVGgHiQtlI0oAQlAG7Hk4lWF0U9g3kcwqy+nCRf9ey/2ihsj9H/9vGhv4/pv2HdLc3Wv8HUQN2AzcM/AZsBjwiq67llWV4ZotTh8Wir1vnIwC9mLi/X3pTP5j9EUbx/Or99c2HOozo9s6uVs1c/N3AvCPPpRvAWOl9+PKCvjDIgnJNAI5CB0gGzNmZoiy30b9XBYUtPi0IZp9qWr5tzgLPowvbhKkJc6z1HTLQrBYM0m/T42cNSgXs2FesMXiqgN+34ulsa2xdAsPL7qQrs2YEEmhApeisni6ZThuuM56NSclddq8yq1OX/aupU+ua3+cauR+cbktylk17Gi+7JjUyGwqpLVgcfRMXMi+yT06CrbEoHV3ZGHvieF6grRMUrlrw9WtIkVHztzDyaCUhHeTqhbyYvMAAuRQyceUVDolmesnhTrOa85BNe6ZoP5vikzomGypFmO4W29fMwKHRMW/fIhvfbMWKXrY5u2GYT61CX60qYzF8eYwzMeU3et4v/pH2f5LErVH86Pt/qyj/tbc38T8eef8v3/Mr+T2/UqvcZ0et4J2fn2HPT3dpJpqZbmRynwHcr6euD1g8sV0v3zpZhK+P3RpxA6gIAXGdLaxZhGHkFJNveTBwDHcdXFYy/AiL/oQSqwt01zwn/oYdqX22hopIUHJb2Rl0Nd8y9PhanU0KSFnJsGyVToVzbO9hDl5U7koKaMvdkbIg0Cd4uUFkOsBz3PDrfGgezP1LH+RkEdjo3t00WVzLe62Cu09ivCrljJfUlxPBeFVY46GaGEG9VkGmq9IBnquSRKhPrzdYg+PC+40etFrEDi6oWHIB1Lo12+zfD7j/fxbZfy35fyd//w86gGz2/0fc/+MJUmfXvygLAjBDL7jSzRmnjNid/xmcv7F8SAS582eUtF7NPY98GGWijTlMoJ+IwaUbp2wi1FD8aGIKiUPQaS8YXaIzqKhd4xA+cFkPOFX3Cj0Y2X6Koq70abmU7gc+nT5xJDUpd9Y/HiuQlbB4Gd93VSPzPVM+vOjphxrPVBquCASyneqahdblQhNNw5XlzZP4wsyvhGsMphdIstOB0ZerB9olUQJc5poDmBpMHYKm0UyQ9++qJCj75AiCDis1FSP9yEI0RszCS4ztqFMmACPa9xYgpZhWqVXXuoKo3Omib8iG3JOagIWMvCByklCvNBNZw4FtpDtZHFvMuQtURctzrneFe3V1xZTCbKwtRrUwRh83HkbJeJQNhCHtlmXprbvcBGMLukG2fpGeXaw5TUcZvjbHV+hp6gMQHvRqaedvdtsvev8Xd2E9tvzf3NtpFuX/TfyfR3lKNvVR4I/di8TonXmcvK2pL3uv9s8O+8IHRaalYurQzu5cY7gTrjjPEVdOt3OoiMgneuBMiAYgJqIGTD4udx5j8hUmoUMqzzCQXR5JmgPwTHqSP+MNp2O2Bt3QViOcDXY/G3+h29n47+ztbFRDnW6Goz9Pa6i/z8Sc0LFIgZt+C9z0wnHT7zLcdD8c/SnBLV2gCxwOxdgTTVXWWP8YbsYUrhUPSQRWrH+jXbj/vd3e2dz//kXo/x7izO8zKwBp3jJOK6JDbDGHTdQ9mLASzEk8/Yy6QB4AJ38mpBV1GNkoWUt1grlwWQONB1mps3BRcjSVYoCpxSVLmsYMn5eEmeosizFVy+aJeUizJGxVLh2GL+KHsc2G0Wjm0+1gFNFJKP3IJfKwVZjMf+YAWDCoTnqzdjaZgtZ3VCnsfpJ+t2FFS+g/3ZhOk+iBOcBV9H+3XeD/djf0/1Hpvz+fzm6QuPkzRoNPDg4F1SUbv9W6oHT6iIz8CwjZhEEi2TwlCLnZbwRU2jdth2uRHTO8OGe5BNW+uamp19dAYv1Zw/VtoFSRnnBa1ZpKlwR3IXHu+nGbH3FbYchywBiPLnUdEPxVbe3s1gCd+IUfv1ONPfV7/Pid2jaqLAmQWtdu1K0b1QZsXRgzkKN/Uk2vuQL81D8N7BEoz7rR4S/skaevn3MyTJfhAly2O3QejitpQz4yGZB8YhOZDgQgSgAY77gMIooxcMMzrhvJps1994+5A7uQF4QRg4LlVz5O4oTl3IOezA5Nth98571Oja+p8vjozRpeydysSntrVg3xC6qxSAWBipB4NOlq6YSQA0Gv6sel9aezJdN2p46PW1O0uhE0S9h8SJRc92pEUtZ9GvFY659X3/Xdz1fGKvm/uZ3j/5t78Gzo/2M8AzYBhtxkmoIvddUoHCmWbQczIlz1Pzb80tf6zG5mYYCksREHU+9PWf9Gq7WT9//c3fh/PtL6P5+7nl2PyOF/qAjjEpUCiYFMPYuDwIuedVmApvcTx/G0ocIynQMr4PgYyVUCbVAayeCaogz47Boq5OeJAYwXyJZ1EiIj7jLCJUfMwCRH2NOZqxL/up+5Vw/dTVSWV7WdaUBnT33Y6W/c+pv9AzXjqKICBuaroh79cvDyYF/tb6vXrYaWtL7OiCGW86zbbhiGpoB87TAZ/xZjcmDSm4O+dqdYZB1CHXYr2mhb/sXlJLBaINtod0OoO4aKY5GsEBC4AI2rQ551oX3GDgmnGg8pgN/aTFjWZjeADJr5rMukb4049eRt5npe8P5Z1+DgwOqMJpDa4BhtdzyeQ8dEiHNbAIWWH0EPTen7DpfLNWTFPeK2AB99HKYD2Aio6y2vLjdmqODmQZOF7SPPujRP0JniGots7WE0sAHOjIY0R2Ywc9BspDEGXn6owLQKHTblwhFk2JDlzbN5Ns/m2TybZ/Nsns2zeTbP5tk8m2fzbJ7Ns3k2z+bZPJtn82yezbN5Ns/muffz/9QGTxgAGAEA"

payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest() == PAYLOAD_SHA256
manifest_bytes = json.dumps(MANIFEST, sort_keys=True, separators=(',', ':')).encode()
assert hashlib.sha256(manifest_bytes).hexdigest() == MANIFEST_SHA256

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True)
with tarfile.open(fileobj=io.BytesIO(payload), mode='r:gz') as tf:
    tf.extractall(WORKSPACE, filter='data')

for item in MANIFEST:
    path = WORKSPACE / item['path']
    data = path.read_bytes()
    assert len(data) == item['bytes']
    assert hashlib.sha256(data).hexdigest() == item['sha256']

for name in ['outputs','metadata','evidence','logs','.runtime']:
    (WORKSPACE/name).mkdir(parents=True, exist_ok=True)
assert not any((WORKSPACE/'evidence').iterdir())
assert not any((WORKSPACE/'outputs').iterdir())
assert not any((WORKSPACE/'metadata').iterdir())
print(f'BOOTSTRAP_FILES={len(MANIFEST)}')
print('BOOTSTRAP_INTEGRITY=PASS')


## Step 3 — Static qualification before GPU load / Kiểm tra tĩnh trước khi nạp GPU

**English**

Run unit/contract tests, Python compilation, shell syntax validation, and notebook/source checks before loading the resident model.

**Tiếng Việt**

Chạy unit/contract test, compile Python, kiểm tra cú pháp shell và các static check trước khi nạp resident model.

### Expected evidence / Kết quả cần thấy

**English**

All test commands succeed and the cell ends with `STATIC_QUALIFICATION=PASS`.

**Tiếng Việt**

Tất cả lệnh test/validation đều thành công và cell kết thúc bằng `STATIC_QUALIFICATION=PASS`.


In [ ]:
import os, subprocess, sys
os.chdir(WORKSPACE)
env = dict(os.environ, PYTHONPATH=str(WORKSPACE/'src'))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True, env=env)
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'src', 'scripts', 'tests'], check=True, env=env)
for script in (WORKSPACE/'scripts').glob('*.sh'):
    subprocess.run(['bash','-n',str(script)], check=True)
print('STATIC_QUALIFICATION=PASS')


## Step 4 — Start the resident BF16 T4×2 worker / Khởi động resident BF16 worker trên T4×2

**English**

Start the internal loopback worker and wait until one BF16 pipeline is resident with the qualified component placement across both T4 GPUs.

**Tiếng Việt**

Khởi động internal loopback worker và chờ một BF16 pipeline resident với component placement đã qualification trên cả hai T4.

### Expected evidence / Kết quả cần thấy

**English**

Readiness JSON shows transformer on GPU0, text encoder + VAE on GPU1, dtype `bfloat16`, then `WORKER_READY=PASS`.

**Tiếng Việt**

Readiness JSON cho thấy transformer trên GPU0, text encoder + VAE trên GPU1, dtype `bfloat16`, sau đó là `WORKER_READY=PASS`.


In [ ]:
import os, secrets, subprocess, sys, time, urllib.request, json
RUNTIME = WORKSPACE/'.runtime'
TOKEN_FILE = RUNTIME/'api_token'
TOKEN_FILE.write_text(secrets.token_urlsafe(32))
os.chmod(TOKEN_FILE, 0o600)
os.environ['ZIMAGE_API_TOKEN'] = TOKEN_FILE.read_text().strip()
env = dict(os.environ, PYTHONPATH=str(WORKSPACE/'src'))
worker_log = open(WORKSPACE/'logs/worker.log','w')
worker = subprocess.Popen([sys.executable,'-m','uvicorn','zimage_server.worker_api:app','--host','127.0.0.1','--port','8101','--log-level','info'], cwd=WORKSPACE, env=env, stdout=worker_log, stderr=subprocess.STDOUT)
(RUNTIME/'worker.pid').write_text(str(worker.pid))

deadline=time.monotonic()+420
while True:
    if worker.poll() is not None:
        raise RuntimeError((WORKSPACE/'logs/worker.log').read_text()[-8000:])
    try:
        with urllib.request.urlopen('http://127.0.0.1:8101/ready', timeout=5) as r:
            ready=json.load(r); break
    except Exception:
        if time.monotonic()>deadline: raise TimeoutError('worker readiness')
        time.sleep(2)
expected={'transformer':0,'text_encoder':1,'vae':1}
assert ready['load']['device_map'] == expected, ready
assert ready['load']['dtype'] == 'bfloat16'
(WORKSPACE/'evidence/runtime-ready.json').write_text(json.dumps(ready, indent=2))
print(json.dumps(ready, indent=2))
print('WORKER_READY=PASS')


## Step 5 — Start the authenticated REST coordinator / Khởi động REST coordinator có xác thực

**English**

Start the CPU-side FastAPI coordinator on port 8090, validate the public health/root routes and authenticated readiness/info routes, and confirm only qualified profiles are exposed.

**Tiếng Việt**

Khởi động FastAPI coordinator phía CPU trên cổng 8090, kiểm tra health/root public cùng readiness/info có xác thực, và xác nhận chỉ expose các profile đã qualification.

### Expected evidence / Kết quả cần thấy

**English**

The runtime info JSON is printed and the cell ends with `COORDINATOR_READY=PASS`.

**Tiếng Việt**

Runtime info JSON được in ra và cell kết thúc bằng `COORDINATOR_READY=PASS`.


In [ ]:
coord_log = open(WORKSPACE/'logs/coordinator.log','w')
coord = subprocess.Popen([sys.executable,'-m','uvicorn','zimage_server.app:app','--host','127.0.0.1','--port','8090','--log-level','info'], cwd=WORKSPACE, env=env, stdout=coord_log, stderr=subprocess.STDOUT)
(RUNTIME/'coordinator.pid').write_text(str(coord.pid))

def get(path, auth=False):
    headers={'Authorization':'Bearer '+os.environ['ZIMAGE_API_TOKEN']} if auth else {}
    req=urllib.request.Request('http://127.0.0.1:8090'+path, headers=headers)
    with urllib.request.urlopen(req, timeout=10) as r:
        return r.status, json.load(r)

deadline=time.monotonic()+30
while True:
    try:
        status, health=get('/health'); break
    except Exception:
        if coord.poll() is not None: raise RuntimeError((WORKSPACE/'logs/coordinator.log').read_text()[-8000:])
        if time.monotonic()>deadline: raise TimeoutError('coordinator readiness')
        time.sleep(1)
assert status==200 and health['status']=='ok'
status, root=get('/')
assert root['type']=='rest-api'
status, ready=get('/ready', auth=True)
assert ready['ready'] is True
status, info=get('/v1/info', auth=True)
assert set(info['profiles']) == {'safe_512','high_768'}
assert 'high_1024' not in info['profiles']
print(json.dumps(info, indent=2))
print('COORDINATOR_READY=PASS')


## Step 6 — Run deterministic 512×512 acceptance / Chạy acceptance 512×512 xác định

**English**

Submit the golden `safe_512` request through the REST API and verify the output checksum against the accepted deterministic image.

**Tiếng Việt**

Gửi golden request `safe_512` qua REST API và đối chiếu checksum output với ảnh deterministic đã được chấp nhận.

### Expected evidence / Kết quả cần thấy

**English**

`GOLDEN_512_DETERMINISM=PASS`; expected PNG SHA-256 is `56e0fca007d3da945de77788b5ab8a0a131b58730d93b624ca65e7c6ca6ea307`.

**Tiếng Việt**

`GOLDEN_512_DETERMINISM=PASS`; SHA-256 PNG mong đợi là `56e0fca007d3da945de77788b5ab8a0a131b58730d93b624ca65e7c6ca6ea307`.


In [ ]:
subprocess.run([sys.executable,'scripts/golden_rest_acceptance.py'], cwd=WORKSPACE, env=env, check=True)
golden=json.loads((WORKSPACE/'metadata/golden-512-seed42.json').read_text())
assert golden['sha256']=='56e0fca007d3da945de77788b5ab8a0a131b58730d93b624ca65e7c6ca6ea307'
print('GOLDEN_512_DETERMINISM=PASS')


## Step 7 — Run five-job queue endurance / Chạy endurance queue 5 job

**English**

Exercise five sequential generation jobs through the bounded single-flight queue and record timing/queue evidence.

**Tiếng Việt**

Chạy năm generation job tuần tự qua bounded single-flight queue và ghi lại timing/queue evidence.

### Expected evidence / Kết quả cần thấy

**English**

The endurance record is printed and the cell ends with `QUEUE_ENDURANCE=PASS`.

**Tiếng Việt**

Endurance record được in ra và cell kết thúc bằng `QUEUE_ENDURANCE=PASS`.


In [ ]:
subprocess.run([sys.executable,'scripts/endurance_queue_acceptance.py'], cwd=WORKSPACE, env=env, check=True)
endurance=json.loads((WORKSPACE/'evidence/endurance-5jobs.json').read_text())
print(json.dumps(endurance, indent=2))
print('QUEUE_ENDURANCE=PASS')


## Step 8 — Verify authentication and error boundaries / Xác minh authentication và error boundary

**English**

Exercise rejected requests for missing/invalid auth, invalid inputs, unsupported profiles/resolutions, queue pressure, worker unavailability, and premature image retrieval.

**Tiếng Việt**

Kiểm tra các request bị từ chối khi thiếu/sai auth, input không hợp lệ, profile/resolution không hỗ trợ, queue pressure, worker unavailable và lấy ảnh quá sớm.

### Expected evidence / Kết quả cần thấy

**English**

The boundary matrix is printed, 1024×1024 is rejected with HTTP 422, and the cell ends with `ERROR_BOUNDARIES=PASS`.

**Tiếng Việt**

Boundary matrix được in ra, 1024×1024 bị từ chối bằng HTTP 422 và cell kết thúc bằng `ERROR_BOUNDARIES=PASS`.


In [ ]:
subprocess.run([sys.executable,'scripts/error_boundary_acceptance.py'], cwd=WORKSPACE, env=env, check=True)
errors=json.loads((WORKSPACE/'evidence/error-boundary.json').read_text())
assert errors['unsupported_1024'] == 422
print(json.dumps(errors, indent=2))
print('ERROR_BOUNDARIES=PASS')


## Step 9 — Run mixed 512 → 768 → 512 qualification / Chạy qualification hỗn hợp 512 → 768 → 512

**English**

Prove that one resident pipeline can serve the qualified low/high/low profile sequence without replacing the model or creating concurrent GPU replicas.

**Tiếng Việt**

Chứng minh một resident pipeline có thể phục vụ chuỗi profile thấp/cao/thấp đã qualification mà không thay model hoặc tạo GPU replica chạy đồng thời.

### Expected evidence / Kết quả cần thấy

**English**

The mixed-profile evidence is printed and the cell ends with `MIXED_PROFILE_QUALIFICATION=PASS`.

**Tiếng Việt**

Mixed-profile evidence được in ra và cell kết thúc bằng `MIXED_PROFILE_QUALIFICATION=PASS`.


In [ ]:
subprocess.run([sys.executable,'scripts/mixed_profile_acceptance.py'], cwd=WORKSPACE, env=env, check=False)
# The first 512 request can legitimately warm allocator state; qualify from captured results.
subprocess.run([sys.executable,'scripts/qualify_mixed_existing.py'], cwd=WORKSPACE, env=env, check=True)
mixed=json.loads((WORKSPACE/'evidence/mixed-512-768-512.json').read_text())
print(json.dumps(mixed, indent=2))
print('MIXED_PROFILE_QUALIFICATION=PASS')


## Step 10 — Optional public HTTPS transport check / Kiểm tra public HTTPS transport tùy chọn

**English**

Optionally expose the coordinator through a temporary Cloudflare Quick Tunnel and run an external REST acceptance check. This is transport evidence only, not runtime authority.

**Tiếng Việt**

Tùy chọn expose coordinator qua Cloudflare Quick Tunnel tạm thời và chạy external REST acceptance. Đây chỉ là transport evidence, không phải runtime authority.

### Expected evidence / Kết quả cần thấy

**English**

Either `PUBLIC_HTTPS_REST_ACCEPTANCE=PASS` or the explicit `PUBLIC_HTTPS_REST_ACCEPTANCE=SKIPPED` marker.

**Tiếng Việt**

Nhận một trong hai marker rõ ràng: `PUBLIC_HTTPS_REST_ACCEPTANCE=PASS` hoặc `PUBLIC_HTTPS_REST_ACCEPTANCE=SKIPPED`.


In [ ]:
RUN_PUBLIC_TUNNEL = False
public_evidence = WORKSPACE / 'evidence/public-tunnel-acceptance.json'

if not RUN_PUBLIC_TUNNEL:
    skipped = {
        'verdict': 'SKIPPED',
        'optional': True,
        'reason': 'Set RUN_PUBLIC_TUNNEL=True to run the external Quick Tunnel gate.'
    }
    public_evidence.write_text(json.dumps(skipped, indent=2))
    print(json.dumps(skipped, indent=2))
    print('PUBLIC_HTTPS_REST_ACCEPTANCE=SKIPPED')
else:
    CF_VERSION = '2026.10.0'
    CF = RUNTIME / 'cloudflared'
    if not CF.exists():
        url = f'https://github.com/cloudflare/cloudflared/releases/download/{CF_VERSION}/cloudflared-linux-amd64'
        urllib.request.urlretrieve(url, CF)
        os.chmod(CF, 0o755)
    tunnel_log = open(WORKSPACE / 'logs/cloudflared.log', 'w')
    tunnel = subprocess.Popen([
        str(CF), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8090'
    ], stdout=tunnel_log, stderr=subprocess.STDOUT)
    (RUNTIME / 'tunnel.pid').write_text(str(tunnel.pid))
    import re
    deadline = time.monotonic() + 30
    public_url = None
    while time.monotonic() < deadline:
        time.sleep(1)
        text = (WORKSPACE / 'logs/cloudflared.log').read_text(errors='replace')
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
        if m:
            public_url = m.group(0)
            break
    if public_url is None:
        raise RuntimeError('Quick Tunnel URL not established')
    (RUNTIME / 'tunnel_url').write_text(public_url)
    subprocess.run([sys.executable, 'scripts/public_tunnel_acceptance.py'], cwd=WORKSPACE, env=env, check=True)
    print('PUBLIC_HTTPS_REST_ACCEPTANCE=PASS')


## Step 11 — Finalize the qualification record / Hoàn tất qualification record

**English**

Aggregate the current-run evidence into the machine-readable summary and require a final PASS verdict. The local REST API remains available until you explicitly stop it.

**Tiếng Việt**

Tổng hợp evidence của run hiện tại thành summary machine-readable và bắt buộc final verdict là PASS. Local REST API vẫn hoạt động cho đến khi bạn chủ động dừng.

### Expected evidence / Kết quả cần thấy

**English**

`FINAL_QUALIFICATION=PASS`. To stop the services after review, run `bash scripts/stop.sh` from the materialized workspace.

**Tiếng Việt**

`FINAL_QUALIFICATION=PASS`. Sau khi review xong, có thể dừng service bằng `bash scripts/stop.sh` từ materialized workspace.


In [ ]:
subprocess.run([sys.executable,'scripts/finalize_evidence.py'], cwd=WORKSPACE, env=env, check=True)
summary=json.loads((WORKSPACE/'evidence/qualification-summary.json').read_text())
assert summary['final_verdict']=='PASS', summary
summary['notebook']={
    'api_only': True,
    'payload_sha256': PAYLOAD_SHA256,
    'payload_manifest_sha256': MANIFEST_SHA256,
    'model_variation':'dangkhoa2016/tongyi-mai-z-image-turbo/PyTorch/default/1',
    'upstream_revision':'f332072aa78be7aecdf3ee76d5c247082da564a6',
}
(WORKSPACE/'evidence/qualification-summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print('FINAL_QUALIFICATION=PASS')
print('REST API remains on http://127.0.0.1:8090')
print('Bearer token remains private in .runtime/api_token; do not publish it.')
print('Stop services with: bash scripts/stop.sh')
